# Covalent and Allosteric Inhibition Identification

**Author** Benjamin Stickland  
**Affiliation** UNC Chapel Hill  
**Date** 2026

In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
from rdkit import Chem, rdBase
from rdkit.Chem import inchi
from rdkit.Chem.MolStandardize import rdMolStandardize
from requests.adapters import HTTPAdapter
from tqdm import tqdm
from urllib3.util.retry import Retry

## Settings

This notebook queries live KLIFS, so annotations can differ from those used in the paper.

The annotations used in the paper are stored in `Data/Binding_Type.csv`. This notebook writes to `Output/Intermediate/Regenerated_Inputs/Binding_Type.csv`. Set `USE_PAPER_INPUTS = False` in notebook 03 to use regenerated inputs from notebooks 01 and 02.

In [2]:
working_dir = Path.cwd()
for project_dir in (working_dir, working_dir.parent, working_dir / "Code"):
    if (project_dir / "Data" / "KIPS_Dataset.csv").is_file():
        break
else:
    raise FileNotFoundError("Could not find Data/KIPS_Dataset.csv")

DATA_DIR = project_dir / "Data"
OUTPUT_FILE = project_dir / "Output" / "Intermediate" / "Regenerated_Inputs" / "Binding_Type.csv"

COMPOUND_COLUMNS = ["Compound ID", "SMILES"]
BINDING_TYPES = {
    (False, False): "Unknown",
    (True, False): "Allosteric",
    (False, True): "Covalent",
    (True, True): "Allosteric + Covalent",
}

KLIFS_API = "https://klifs.net/api_v2"
KLIFS_MAX_WORKERS = 16
KLIFS_TIMEOUT = 60
KLIFS_RETRY = Retry(
    total=3,
    backoff_factor=0.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)
KLIFS_EMPTY_RESPONSES = {
    "structures_list": "KLIFS error: An unknown kinase ID was provided",
    "ligands_list": "KLIFS error: The provided kinase ID(s) is/are invalid",
}

## Prepare structure keys

Compare standardized InChIKeys across sources. Use RDKit to select the parent fragment, uncharge it, and sanitize the molecule. Keep the original KIPS SMILES for export.

In [3]:
def standardize_inchikey(smiles: str, uncharger: rdMolStandardize.Uncharger) -> str | None:
    """Return the standardized InChIKey, or None if the structure cannot be converted."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        mol = rdMolStandardize.FragmentParent(mol)
        mol = uncharger.uncharge(mol)
        Chem.SanitizeMol(mol)
        return inchi.MolToInchiKey(mol) or None
    except (ValueError, RuntimeError):
        return None


def compute_inchikeys(smiles: pd.Series, source: str) -> pd.Series:
    """Convert each distinct SMILES once and return keys in the original row order."""
    uncharger = rdMolStandardize.Uncharger()
    keys = {}
    with rdBase.BlockLogs():
        for structure in tqdm(smiles.dropna().unique(), desc=source):
            keys[structure] = standardize_inchikey(structure, uncharger)

    inchikeys = smiles.map(keys)
    invalid_count = inchikeys.isna().sum()
    if invalid_count:
        print(f"{source} has {invalid_count} rows with missing or invalid structures.")
    return inchikeys

## Load KIPS and local references

Use rows marked `allosteric` in the published dataset and all structures in CovalentInDB. A missing or invalid structure in a reference dataset contributes no match.

In [4]:
compounds = pd.read_csv(
    DATA_DIR / "KIPS_Dataset.csv",
    usecols=COMPOUND_COLUMNS,
    dtype="string",
)

compound_keys = compute_inchikeys(compounds["SMILES"], "KIPS")
invalid_compounds = compound_keys.isna()
if invalid_compounds.any():
    invalid_ids = compounds.loc[invalid_compounds, "Compound ID"].tolist()
    raise ValueError(f"KIPS structures could not be standardized for compound IDs {invalid_ids}.")

allosteric_reference = pd.read_csv(
    DATA_DIR / "allosteric_PKIs_2023.tsv",
    sep="\t",
    usecols=["SMILES", "Mechanism"],
    dtype="string",
)
allosteric_rows = allosteric_reference["Mechanism"].str.strip().str.casefold().eq("allosteric")
published_smiles = allosteric_reference.loc[allosteric_rows, "SMILES"]
published_keys = set(compute_inchikeys(published_smiles, "Published allosteric").dropna())

covalent_reference = pd.read_csv(DATA_DIR / "CovInDB_All.csv", usecols=["SMILES"], dtype="string")
covalent_keys = set(compute_inchikeys(covalent_reference["SMILES"], "CovalentInDB").dropna())

CovalentInDB: 100%|██████████| 8281/8281 [00:15<00:00, 529.10it/s]


## Retrieve human KLIFS structures and ligands

In [5]:
def fetch_klifs_records(endpoint: str, **params: str | int) -> list[dict]:
    with requests.Session() as session:
        session.mount("https://", HTTPAdapter(max_retries=KLIFS_RETRY))
        response = session.get(f"{KLIFS_API}/{endpoint}", params=params, timeout=KLIFS_TIMEOUT)

    if endpoint in KLIFS_EMPTY_RESPONSES and response.status_code == 400:
        if response.json() == [400, KLIFS_EMPTY_RESPONSES[endpoint]]:
            return []

    response.raise_for_status()
    records = response.json()
    if not isinstance(records, list) or any(not isinstance(record, dict) for record in records):
        raise RuntimeError(f"Unexpected KLIFS {endpoint} response for {params}")
    return records


def fetch_kinase_records(endpoint: str, kinase_ids: list[int]) -> list[dict]:
    records = []
    empty_count = 0
    with ThreadPoolExecutor(max_workers=KLIFS_MAX_WORKERS) as pool:
        requests_by_kinase = {
            pool.submit(fetch_klifs_records, endpoint, kinase_ID=kinase_id): kinase_id
            for kinase_id in kinase_ids
        }
        for request in tqdm(
            as_completed(requests_by_kinase), total=len(requests_by_kinase), desc=f"KLIFS {endpoint}"
        ):
            try:
                kinase_records = request.result()
            except Exception as exc:
                kinase_id = requests_by_kinase[request]
                for pending_request in requests_by_kinase:
                    pending_request.cancel()
                raise RuntimeError(f"KLIFS {endpoint} failed for kinase {kinase_id}.") from exc

            if not kinase_records:
                empty_count += 1
            records.extend(kinase_records)

    kinases_with_records = len(kinase_ids) - empty_count
    print(f"KLIFS {endpoint} returned records for {kinases_with_records}/{len(kinase_ids)} kinases" f"\n")
    return records


def require_klifs_columns(table: pd.DataFrame, columns: set[str], endpoint: str) -> None:
    if table.empty:
        raise RuntimeError(f"KLIFS {endpoint} returned no records.")
    missing_columns = columns.difference(table.columns)
    if missing_columns:
        raise RuntimeError(f"KLIFS {endpoint} is missing columns {sorted(missing_columns)}.")

## Resolve KLIFS allosteric ligands

Find each structure's allosteric ligand in the ligand catalog by its KLIFS ligand ID or PDB code. Standardize the SMILES for the matched ligands with the same procedure used for the local data.

In [6]:
def clean_klifs_ids(values: pd.Series, *, numeric: bool = False) -> pd.Series:
    cleaned = values.astype("string").str.strip()
    if numeric:
        cleaned = cleaned.str.replace(r"\.0$", "", regex=True)
    return cleaned.replace({"0": pd.NA, "0.0": pd.NA, "-": pd.NA, "": pd.NA})


def select_allosteric_ligands(structures: pd.DataFrame, ligands: pd.DataFrame) -> pd.DataFrame:
    allosteric_ids = set(clean_klifs_ids(structures["allosteric_ligand_ID"], numeric=True).dropna())
    allosteric_codes = set(clean_klifs_ids(structures["allosteric_ligand"]).dropna())
    ligand_ids = clean_klifs_ids(ligands["ligand_ID"], numeric=True)
    ligand_codes = clean_klifs_ids(ligands["PDB-code"])

    unresolved_ids = allosteric_ids.difference(ligand_ids.dropna())
    unresolved_codes = allosteric_codes.difference(ligand_codes.dropna())
    if unresolved_ids or unresolved_codes:
        print(
            "Warning: KLIFS allosteric references missing from the ligand catalog. "
            f"Ligand IDs {sorted(unresolved_ids)}, PDB codes {sorted(unresolved_codes)}."
        )

    matches = ligand_ids.isin(allosteric_ids) | ligand_codes.isin(allosteric_codes)
    return ligands.loc[matches]


kinases = pd.DataFrame(fetch_klifs_records("kinase_names", species="Human"))
require_klifs_columns(kinases, {"kinase_ID"}, "kinase_names")
kinase_ids = kinases["kinase_ID"].dropna().astype(int).drop_duplicates().tolist()
if not kinase_ids:
    raise RuntimeError("KLIFS returned no human kinase IDs")

structures = pd.DataFrame(fetch_kinase_records("structures_list", kinase_ids))
require_klifs_columns(structures, {"allosteric_ligand_ID", "allosteric_ligand"}, "structures_list")

ligands = pd.DataFrame(fetch_kinase_records("ligands_list", kinase_ids))
require_klifs_columns(ligands, {"ligand_ID", "PDB-code", "SMILES"}, "ligands_list")

allosteric_ligands = select_allosteric_ligands(structures, ligands)
klifs_smiles = allosteric_ligands["SMILES"].astype("string")
klifs_keys = set(compute_inchikeys(klifs_smiles, "KLIFS allosteric").dropna())

KLIFS structures_list: 100%|██████████| 555/555 [00:24<00:00, 23.07it/s]


KLIFS structures_list returned records for 329/555 kinases



KLIFS ligands_list: 100%|██████████| 555/555 [00:28<00:00, 19.50it/s]


KLIFS ligands_list returned records for 297/555 kinases



KLIFS allosteric: 100%|██████████| 322/322 [00:00<00:00, 618.85it/s]


## Assign binding types

Compounds can match both mechanisms. `Unknown` when neither mechanism has a match in the queried sources.

In [15]:
published_matches = compound_keys.isin(published_keys)
klifs_matches = compound_keys.isin(klifs_keys)
covalent_matches = compound_keys.isin(covalent_keys)
allosteric_matches = published_matches | klifs_matches

annotations = compounds.loc[:, COMPOUND_COLUMNS].assign(
    Type=[BINDING_TYPES[flags] for flags in zip(allosteric_matches, covalent_matches)]
)

print(f"KIPS matches in the published allosteric dataset: {published_matches.sum()}")
print(f"KIPS matches in KLIFS: {klifs_matches.sum()}")
print(f"KIPS matches in CovalentInDB: {covalent_matches.sum()}")
print(f"\nBinding types:\n{annotations['Type'].value_counts().to_string(header=False)}")

KIPS matches in the published allosteric dataset: 6
KIPS matches in KLIFS: 8
KIPS matches in CovalentInDB: 5

Binding types:
Unknown                  1537
Allosteric                 11
Covalent                    4
Allosteric + Covalent       1


## Export the binding annotations

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
annotations.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print(f"Saved {len(annotations):,} annotations to {OUTPUT_FILE}")